<a href="https://colab.research.google.com/github/GurneeshBudhiraja/ArangoDB-Hackathon/blob/main/ArangoDB_Hackathon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Testing Application using Arango DB and Synthea Dataset using Google Colab's T4 Runtime

In [1]:
# Required packages
!pip install nx-arangodb
!pip install arango-datasets
!nvidia-smi
!nvcc --version
!pip install nx-cugraph-cu12 --extra-index-url https://pypi.nvidia.com
!pip install google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.8/67.8 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 69.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.6/114.6 kB 9.3 MB/s eta 0:00:00
  Attempting uninstall: networkx
    Found existing installation: networkx 3.4.2
    Uninstalling networkx-3.4.2:
      Successfully uninstalled networkx-3.4.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.5.1+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.5.1+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cuda-cupti-cu12 12.5.82 which

In [2]:
# Import required modules
import json
from google.colab import userdata

import networkx as nx
import nx_arangodb as nxadb

from arango import ArangoClient
from arango_datasets import Datasets


# Gemini SDK Packages
from google import genai

[21:12:18 +0000] [INFO]: NetworkX-cuGraph is available.
INFO:nx_arangodb:NetworkX-cuGraph is available.
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:502: UserWarning: <built-in function any> is not a Python type (it may be an instance of an object), Pydantic will allow any object with no validation since we cannot even enforce that the input is an instance of the given type. To get rid of this error wrap the type with `pydantic.SkipValidation`.
  warn(


In [3]:
# Imports the secrets from the Colab notebook
ARANGO_HOST = userdata.get("ARANGO_HOST")
ARANGO_PASSWORD = userdata.get("ARANGO_PASSWORD")
ARANGO_USERNAME = userdata.get("ARANGO_USERNAME")
GEMINI_API = userdata.get("GEMINI_API_KEY")
FLASH_MODEL = "gemini-2.0-flash"


# Creates a db connection
db = ArangoClient(hosts=ARANGO_HOST).db(username=ARANGO_USERNAME, password=ARANGO_PASSWORD, verify=True)
gemini_client = genai.Client(
    api_key=GEMINI_API
)

In [4]:
# Creates the dataset connection using the db object
datasets = Datasets(db)

DATASET_NAME = "SYNTHEA_P100"

# Conditionally Loads the Synthea P100 dataset in Arango
if not db.has_graph(DATASET_NAME):
  datasets.load(dataset_name=DATASET_NAME)
else:
  print(f"{DATASET_NAME} is already in ArangoDB.")

Output()

SYNTHEA_P100 is already in ArangoDB.


In [5]:
# Connects with the Graph in ArangoDB
graph = None
if db.has_graph(DATASET_NAME):
  graph = nxadb.Graph(name=DATASET_NAME, db=db)
else:
  print("Graph does not exist in Arango DB")

print(graph)

[21:12:23 +0000] [INFO]: Graph 'SYNTHEA_P100' exists.
INFO:nx_arangodb:Graph 'SYNTHEA_P100' exists.
[21:12:23 +0000] [INFO]: Default node type set to 'allergies'
INFO:nx_arangodb:Default node type set to 'allergies'


Graph named 'SYNTHEA_P100' with 145514 nodes and 311701 edges


In [27]:

from pydantic import BaseModel
# Tools for the agent

# Tool to decide whether to use AQL, NetworkX or Hybrid to resolve the user query
def decide_method(user_query:str) -> str:
  """
  This function is responsible to decide whether to use AQL or NetworkX or Combined algorithms to further answer the user query.

  Args:
    user_query: The question asked by the user.

  Returns:
    Returns a string with the prefered method to use for the user_query
  """
  print("================= USING THE decide_method TOOL =================")
  system_instruction = f"""You are an expert in choosing the best method to query a graph database.
        You must choose one of the following methods:
        - aql: For direct database queries and relationship traversals.
        - networkX: For graph algorithm calculations.
        - hybrid: For combining AQL and NetworkX.

        Return ONLY a JSON object in the following format:

        {{
        "method": "method_name"
        }}

        Where 'method_name' is one of 'aql', 'networkX', or 'hybrid'. Do NOT return anything else.
        """
  response = gemini_client.models.generate_content(
      model=FLASH_MODEL,
      config={
          "system_instruction": system_instruction
      },
      contents=[user_query]
  )
  print(response.text)
  return response.text


class AQLSchema(BaseModel):
  query:str

# Tool to generate aql queries
def aql_generator(user_query: str) -> dict[str, str]:
  """
  This is used to generate the AQL queries using the user query

  Args:
    user_query: The query asked by the user

  Returns:
    Returns a dict with a key 'query' and AQL query which will be a string
  """
  print("================= USING THE aql_generator TOOL =================")
  aql_generator_system_prompt = """
  You are the ai assistant who will generate AQL queries. The ArangoDB has Synthea_P100 dataset with the following vertex and edge collections:

  Vertex Collections:
    allergies
    careplans
    conditions
    devices
    encounters
    imaging_studies
    immunizations
    medications
    observations
    organizations
    patients
    payers
    procedures
    providers
    supplies

  Edge Collections:
    encounters_to_allergies
    encounters_to_careplans
    encounters_to_conditions
    encounters_to_devices
    encounters_to_imaging_studies
    encounters_to_immunizations
    encounters_to_medications
    encounters_to_observations
    encounters_to_procedures
    encounters_to_supplies
    organizations_to_encounters
    organizations_to_providers
    patients_to_allergies
    patients_to_careplans
    patients_to_conditions
    patients_to_devices
    patients_to_encounters
    patients_to_imaging_studies
    patients_to_immunizations
    patients_to_medications
    patients_to_observations
    patients_to_procedures
    patients_to_supplies
    payers_to_encounters
    payers_to_medications
    providers_to_encounters

  Your job is to generate an AQL query based on the user's query. Make sure the query is correct and achieves the purpose and gets the info that is needed. Make sure the AQL query should work on the Synthea_P100 dataset and make sure that the collection names, field names, and any other thing you mention is compatible with the dataset that is in the ArangoDB dataset.
  """
  response = gemini_client.models.generate_content(
      model="gemini-1.5-pro",
      config={
          "system_instruction": aql_generator_system_prompt,
          "response_mime_type": "application/json",
          "response_schema":AQLSchema
      },
      contents=[user_query]
  )
  return (json.loads(response.text))


def AQL_executor(aql_query:str):
  """
  This is used to executed the queries that are in the AQL format using the arangoDB client

  Args:
    aql_query: This is the AQL query that can be executed in the ArangoDB

  Returns:
    This returns the list of the data from the ArangoDB. The empty list means that nothing related to that query is available
  """
  print("================= USING THE AQL_executor TOOL =================")
  print(f"AQL Query:\n{aql_query}")
  cursor = db.aql.execute(aql_query)
  return list(cursor)


# Tool to generate networkX queries
def networkX_generator(user_query:str) -> dict[str, str]:
  """
  This is used to generate the NetworkX queries using the user query

  Args:
    user_query: The query asked by the user

  Returns:
    Returns a dict with a key 'query' which will be NetworkX query which will be a string
  """
  print("================= USING THE networkX_generator TOOL =================")
  return {"query":"This is a sample networkX query"}

class QueryEnhancerSchema(BaseModel):
  refined_query:str
def query_enhancer(user_query:str)->str:
  """
  This tool is to enhance and polish the initial user query.

  Args:
  user_query: The initial user_query entered by the user

  Returns:
  The string with the refined user_query to be used for the further tools.
  """
  print("================= USING THE query_enhancer TOOL =================")

  query_enhancer_prompt = """ Your job is to take the current user query and modify it so that it could fetch the data from the arangoDB. The data in the arango db is Synthea_P100 so modify the initial query so that it could fetch the generic overall data asked by the user.

  Example, if the user entered find the name of the patient with the patient id patients/xxxxx-yyyyy-zzzzz, then since in the synthea dataset there is no such field as Name but FIRST and LAST fields exists that contains the name of the patient. Your job is to modify the query so that it is able to fetch the generic data about that patient like modifying it to 'get the data of the patient whose id is patients/xxxxx-yyyyy-zzzzz.'

  Make sure while modifying the query do not modify the significant details like patient details or any other quentissential details from the initial user query.
  """
  json_repsonse = gemini_client.models.generate_content(
      model=FLASH_MODEL,
      config={
          "system_instruction": query_enhancer_prompt,
          "response_mime_type": "application/json",
          "response_schema":QueryEnhancerSchema
      },
      contents=[user_query]
  )
  response = json.loads(json_repsonse.text)
  print(f"================= The new query is =================\n{response}")
  return response["refined_query"] or ""

In [33]:
tools = [decide_method,aql_generator,networkX_generator, AQL_executor,query_enhancer]
system_instructions = """You are the smart agent whose main job is to use the required tools to help answer the user query when the data that is stored is Synthea_P100. These are the tools you have access to:

Please make sure to run the query enhancer tool polishing the initial user query and then use that modifed user query for other tools. For now your only job is to call the required tools to let the user know what method/approach would be best to go for the given user_query. Also, if a tool is available to generate and execute a query for the method selected, use that tool as well and then return the final response to the user.
Please note there would be instances where the response of one tool would heavily depend on the response of the previous tool. So make sure to run the tools in the proper order and also you may need to run more than one tool to reach to the final answer.

You are allowed to modify the user query so that when you pass it to the tools, it fetches the relevant data for you to see and answer the question.

If a query wants multiple info about the user from different collections, or maybe rows, then you are free to split the query as many times as possible to reach to the final answer.
"""

In [34]:

config = {
    'tools': tools,
    'system_instruction': system_instructions,
}


In [41]:
user_input = input("Enter your query: ").strip()
agent_response = gemini_client.models.generate_content(
    model=FLASH_MODEL,
    config=config,
    contents=[user_input]
)
print(f"Agent response is:")
print(agent_response.text)

Enter your query: what is the name and the careplans offered to the patient whose patient id is patients/4e1b0380-6e73-9975-ca00-22b2bc06494a
================= USING THE query_enhancer TOOL =================
================= The new query is =================
{'refined_query': 'get the first name, last name, and the care plans of the patient whose id is patients/4e1b0380-6e73-9975-ca00-22b2bc06494a'}
================= USING THE decide_method TOOL =================
```json
{
        "method": "aql"
}
```
================= USING THE aql_generator TOOL =================
================= USING THE AQL_executor TOOL =================
AQL Query:
FOR p IN patients FILTER p._id == "patients/4e1b0380-6e73-9975-ca00-22b2bc06494a" LET carePlans = (FOR cp IN careplans FILTER cp.PATIENT == p._id RETURN cp) RETURN {firstName: p.FIRST, lastName: p.LAST, carePlans: carePlans}
Agent response is:
The patient's first name is Christene303, the last name is Hane680, and there are no care plans offered to